# Détection des incidents de cassettes sur les terminaux

## À quoi sert ce notebook
Repérer, sur **tout le parc** et **heure par heure**, les terminaux dont une cassette se comporte anormalement :
inversion des billets de 10 € et de 20 €, erreur de paramétrage, cassette vide ou bloquée, cassette inutilisée
qui se remet à distribuer. Une inversion de cassettes est rare, coûteuse, et ne se voit pas dans le
montant des retraits : le terminal enregistre ce qu'il **croit** distribuer.

## Mode d'emploi
1. **Recette SQL** `cassettes_terminal_heure` : la lancer (elle prend automatiquement les 70 derniers jours).
2. **Ce notebook** : modifier uniquement la **cellule 1 (Réglages)** si besoin, puis *Kernel → Restart & Run All*.
3. **Lire les résultats** dans cet ordre :
   - **section 3** : la date maximale des données (la période voulue est-elle couverte ?) ;
   - **section 6** : la liste des anomalies, de la plus urgente à la moins urgente, avec une phrase par anomalie ;
   - **section 10** : le zoom sur les terminaux et la période du cas connu ;
   - **section 8** : les graphiques des billets sortis par coupure.
   - **section 11** : le zoom sur 1 à 3 terminaux (réglage `ZOOM_TERMINAL`) : étude d'inversion refaite et bandeaux d'impact.

## Vocabulaire
- **Cassette 1 à 4** : les 4 bacs de billets d'un terminal. La valeur entre parenthèses, par exemple « cassette 2
  (10 €) », est la coupure que cette cassette contient d'habitude. Elle change d'un terminal à l'autre.
- **Habitude** : ce que le terminal fait normalement, mesuré sur les 4 semaines qui précèdent la période analysée,
  au même jour de semaine et aux mêmes heures.
- **Billets sortis / habituels (x ratio)** : « 459 / 148 (x3,1) » = la cassette a distribué 459 billets
  alors qu'elle en distribue 148 d'habitude, soit 3,1 fois plus.
- **Cassette inactive** : cassette qui ne distribue plus en temps normal (ex. 5 €). Elle est ignorée, sauf si
  elle se remet à distribuer.
- **Chargement** : remontée de l'encaisse théorique, quand le transporteur recharge le terminal. Une inversion se
  produit toujours à un chargement.
- **Priorité** : haute = à vérifier tout de suite (inversion possible, paramétrage, cassette réactivée) ;
  moyenne = cassette vide, bloquée ou sur-sollicitée ; basse = signal isolé.

## Contrôles automatiques
Le notebook s'arrête avec un message clair si une colonne manque dans le dataset, et prévient si les données
ne couvrent pas la période de référence ou la période du cas connu.

## 0. Préparation (ne rien modifier)

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

try:
    from scipy.stats import chi2
except ImportError:          # scipy absent : seuil théorique remplacé par le maximum de référence seul
    chi2 = None
pd.set_option("display.max_columns", 60)

## 1. Réglages — la seule cellule à modifier

In [ ]:
# ============================================================================================================
# À MODIFIER SI BESOIN
# ============================================================================================================
DATASET_CASSETTES = "cassettes_terminal_heure"     # nom du dataset produit par la recette cassettes_terminal_heure
DEBUT_ANALYSE = None                          # None = automatique (14 derniers jours) ; ou "AAAA-MM-JJ"
JOURS_ANALYSE = 14                            # nombre de jours examinés quand DEBUT_ANALYSE = None
CAS_CONNU = None                              # cas confirmé pour valider la méthode, ex. :
# CAS_CONNU = {"debut": "AAAA-MM-JJ HH:MM", "fin": "AAAA-MM-JJ HH:MM",
#              "prefixe_site": "123", "perte_declaree": 5000, "jours_autour": 2}
DATASET_SORTIE_EVENEMENTS = None              # ex. "incidents_cassettes" (dataset à créer dans le Flow)
NB_ANOMALIES_AFFICHEES = 40                   # nombre de lignes affichées en section 6
ZOOM_TERMINAL = []                                 # section 11 : 1 à 3 terminaux à zoomer, ex. ["123921"] ou ["123921", "123922", "123923"]
                                              # (num_automate ou code site) ; vide = terminal du cas connu + inversions possibles
ZOOM_DEBUT = None                             # début du zoom "AAAA-MM-JJ" ; None = 2 jours avant la période suspecte
ZOOM_FIN = None                               # fin du zoom "AAAA-MM-JJ" ; None = 2 jours après la période suspecte
ZOOM_PERIODE_SUSPECTE = None                  # zone grisée étudiée, ex. ("AAAA-MM-JJ HH:MM", "AAAA-MM-JJ HH:MM") ;
                                              # None = période du cas connu
AFFICHER_DETAIL = False                       # True = afficher aussi les tableaux techniques complets

# ============================================================================================================
# PARAMÈTRES DE LA MÉTHODE (ne pas modifier sans raison)
# ============================================================================================================
JOURS_REFERENCE = 28          # habitude du terminal : 4 semaines avant DEBUT_ANALYSE
FENETRE_HEURES = 24           # fenêtre glissante examinée
MIN_BILLETS_FENETRE = 100     # en dessous, la répartition n'est pas interprétable
ECART_PART_MIN = 0.10         # écart minimal de part d'une position (10 points)
MARGE_REFERENCE = 1.5         # le signal doit dépasser 1,5 fois le maximum atteint en référence
MIN_REPETES = 5               # nombre minimal de retraits répétés dans la fenêtre
PART_ACTIVE_MIN = 0.01        # une cassette sortant moins de 1 % des billets en référence est considérée inactive
RATIO_SUR = 2.0               # cassette sur-sollicitée : au moins 2 fois ses billets habituels sur 24 h...
MIN_EXCES_BILLETS = 60        # ... et au moins 60 billets de plus que d'habitude
RATIO_SOUS = 0.3              # cassette sous-utilisée : au plus 30 % de ses billets habituels...
MIN_BILLETS_HABITUELS = 60    # ... quand elle en sort d'habitude au moins 60 sur 24 h
MIN_BILLETS_REACTIVATION = 20 # cassette inactive qui sort au moins 20 billets sur 24 h
POSITIONS = [1, 2, 3, 4]

## 2. Outils d'affichage (ne rien modifier)

In [ ]:
BLEU, GRIS, GRIS_CLAIR, ORANGE, ENCRE = "#2a78d6", "#8a8984", "#d9d8d3", "#eb6834", "#0b0b0b"
COULEUR_COUPURE = {10: ORANGE, 20: BLEU, 50: GRIS, 5: ENCRE, 100: ENCRE}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#ececea", "grid.linewidth": 0.8, "axes.edgecolor": "#b5b4ae",
                     "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9})
CSS = """<style>
.kpi-grille{display:flex;flex-wrap:wrap;gap:10px;margin:6px 0 14px}
.kpi{border:1px solid #e3e2de;border-radius:8px;padding:10px 14px;min-width:150px;background:#fcfcfb}
.kpi .v{font-size:22px;font-weight:700;color:#0b0b0b}.kpi .l{font-size:12px;color:#52514e}
table.t{border-collapse:collapse;font-size:12px;margin:4px 0 12px}
table.t th{background:#f3f2ef;color:#0b0b0b;text-align:left;padding:5px 9px;border-bottom:2px solid #d9d8d3}
table.t td{padding:4px 9px;border-bottom:1px solid #ecebe8;text-align:right}
table.t td.txt{text-align:left}
.note{color:#52514e;font-size:12.5px;margin:2px 0 10px;max-width:900px}
</style>"""


def echapper(texte):
    """Texte sûr dans du HTML."""
    return str(texte).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def formater(v):
    """Nombre lisible : séparateur de milliers, 1 décimale au plus."""
    if isinstance(v, (bool, np.bool_)):
        return "oui" if v else "non"
    if isinstance(v, (int, np.integer)):
        return "{:,}".format(int(v)).replace(",", " ")
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return "–"
        return "{:,.1f}".format(v).replace(",", " ") if abs(v) < 1000 else "{:,.0f}".format(v).replace(",", " ")
    if isinstance(v, pd.Timestamp):
        return v.strftime("%d/%m %Hh")
    return "" if v is None else str(v)


def afficher_titre(titre, note=None):
    html = CSS + "<h4 style='margin:10px 0 2px'>{}</h4>".format(titre)
    if note:
        html += "<div class='note'>{}</div>".format(note)
    display(HTML(html))


def afficher_chiffres(chiffres, titre=None):
    tuiles = "".join("<div class='kpi'><div class='v'>{}</div><div class='l'>{}</div></div>".format(
        echapper(formater(v)), k) for k, v in chiffres.items())
    display(HTML(CSS + ("<h4 style='margin:10px 0 2px'>{}</h4>".format(titre) if titre else "") +
                 "<div class='kpi-grille'>{}</div>".format(tuiles)))


def afficher_tableau(table, titre=None, note=None, max_lignes=40):
    if titre:
        afficher_titre(titre, note)
    if table is None or len(table) == 0:
        display(HTML(CSS + "<div class='note'>Aucune ligne.</div>"))
        return
    entete = "".join("<th>{}</th>".format(echapper(c)) for c in table.columns)
    corps = []
    for _, ligne in table.head(max_lignes).iterrows():
        cellules = "".join("<td class='{}'>{}</td>".format("txt" if isinstance(ligne[c], str) else "",
                                                          echapper(formater(ligne[c]))) for c in table.columns)
        corps.append("<tr>{}</tr>".format(cellules))
    display(HTML(CSS + "<table class='t'><tr>{}</tr>{}</table>".format(entete, "".join(corps))))

## 3. Chargement des données et vérification de la période
Chaque terminal reçoit toutes les heures de la période : une heure sans retrait vaut 0 (elle n'existe pas dans la
recette). Les valeurs déclarées restent vides quand aucun billet n'est sorti.

In [ ]:
def lire_dataset(nom):
    import dataiku
    return dataiku.Dataset(nom).get_dataframe()


brut = lire_dataset(DATASET_CASSETTES)
OBLIGATOIRES = (["num_automate", "date_jour", "heure", "nb_retraits", "nb_retraits_repetes_10min"] +
                ["nb_billets_pos{}".format(k) for k in POSITIONS] + ["val_max_pos{}".format(k) for k in POSITIONS])
manquantes = [c for c in OBLIGATOIRES if c not in brut.columns]
if manquantes:
    raise ValueError("Colonnes absentes du dataset {} : {}. Vérifier que la recette cassettes_terminal_heure est à jour "
                     "et relancée.".format(DATASET_CASSETTES, ", ".join(manquantes)))
for c in ["nb_montants_finissant_10", "montant_total"] + ["nb_retraits_pos{}".format(k) for k in POSITIONS] + \
        ["val_min_pos{}".format(k) for k in POSITIONS]:
    if c not in brut.columns:
        brut[c] = np.nan
jour = pd.to_datetime(brut["date_jour"].astype(str).str[:10], errors="coerce")
heure = pd.to_numeric(brut["heure"], errors="coerce")
illisibles = int((jour.isna() | heure.isna() | (heure < 0) | (heure > 23)).sum())
brut = brut[jour.notna() & heure.between(0, 23)].copy()
brut["ts"] = jour[brut.index] + pd.to_timedelta(heure[brut.index].astype(int), unit="h")
if illisibles:
    afficher_titre("{} ligne(s) écartée(s) : date ou heure illisible".format(illisibles))
COMPTES = (["nb_retraits", "montant_total", "nb_montants_finissant_10", "nb_retraits_repetes_10min"] +
           ["nb_billets_pos{}".format(k) for k in POSITIONS] + ["nb_retraits_pos{}".format(k) for k in POSITIONS])
VALEURS = ["val_min_pos{}".format(k) for k in POSITIONS] + ["val_max_pos{}".format(k) for k in POSITIONS]
COMPTES_ENCAISSE = ["nb_recharges", "montant_recharge", "nb_incoherences_encaisse"]
VALEURS_ENCAISSE = ["encaisse_min_heure", "encaisse_max_heure"]
AVEC_ENCAISSE = all(c in brut.columns for c in COMPTES_ENCAISSE + VALEURS_ENCAISSE)
for c in COMPTES_ENCAISSE + VALEURS_ENCAISSE:
    if c not in brut.columns:
        brut[c] = 0 if c in COMPTES_ENCAISSE else np.nan
COMPTES = COMPTES + COMPTES_ENCAISSE
VALEURS = VALEURS + VALEURS_ENCAISSE
for c in COMPTES + VALEURS:
    brut[c] = pd.to_numeric(brut[c], errors="coerce")
brut[COMPTES] = brut[COMPTES].fillna(0)
for c in ["code_site_implantation", "cd_regate", "code_postal", "ville"]:
    if c not in brut.columns:
        brut[c] = ""
brut["num_automate"] = brut["num_automate"].astype(str)

FIN = brut["ts"].max()      # dernière heure présente : rien n'est inventé après la fin des données
if DEBUT_ANALYSE:
    DEBUT = pd.Timestamp(DEBUT_ANALYSE)
else:
    DEBUT = (FIN - pd.Timedelta(days=JOURS_ANALYSE)).floor("D")
    if CAS_CONNU:   # la période analysée doit contenir le cas connu (avec sa marge)
        DEBUT = min(DEBUT, pd.Timestamp(CAS_CONNU["debut"]).floor("D") -
                    pd.Timedelta(days=CAS_CONNU.get("jours_autour", 2)))
DEBUT_REF = DEBUT - pd.Timedelta(days=JOURS_REFERENCE)
PREMIERE_DATE = brut["ts"].min()
brut = brut[(brut["ts"] >= DEBUT_REF) & (brut["ts"] <= FIN)]
ALERTES_DONNEES = []
if PREMIERE_DATE > DEBUT_REF + pd.Timedelta(days=1):
    ALERTES_DONNEES.append("Les données commencent le {} : la référence ({} jours avant le {}) est incomplète. "
                           "Allonger la période de la recette SQL.".format(PREMIERE_DATE.strftime("%d/%m/%Y"),
                                                                            JOURS_REFERENCE, DEBUT.strftime("%d/%m")))
if CAS_CONNU and pd.Timestamp(CAS_CONNU["debut"]) < PREMIERE_DATE:
    ALERTES_DONNEES.append("Les données commencent le {} : le cas connu ({}) est plus ancien. Relancer la recette SQL "
                           "avec une période qui le contient.".format(PREMIERE_DATE.strftime("%d/%m/%Y"),
                                                                       pd.Timestamp(CAS_CONNU["debut"]).strftime("%d/%m/%Y")))
if CAS_CONNU and pd.Timestamp(CAS_CONNU["fin"]) > FIN:
    ALERTES_DONNEES.append("Les données s'arrêtent le {} : la fin du cas connu ({}) n'est pas couverte.".format(
        FIN.strftime("%d/%m/%Y %Hh"), pd.Timestamp(CAS_CONNU["fin"]).strftime("%d/%m/%Y %Hh")))

infos = brut.groupby("num_automate")[["code_site_implantation", "cd_regate", "code_postal", "ville"]].agg(
    lambda s: s.dropna().astype(str).iloc[-1] if s.notna().any() else "")
heures = pd.date_range(DEBUT_REF, FIN, freq="H")
index = pd.MultiIndex.from_product([sorted(brut["num_automate"].unique()), heures], names=["num_automate", "ts"])
grille = brut.set_index(["num_automate", "ts"])[COMPTES + VALEURS].reindex(index)
grille[COMPTES] = grille[COMPTES].fillna(0)
grille = grille.reset_index()

afficher_chiffres({"terminal": grille["num_automate"].nunique(), "heures par terminal": len(heures),
                   "retraits": int(grille["nb_retraits"].sum()),
                   "chargements repérés": int(grille["nb_recharges"].sum()) if AVEC_ENCAISSE else "colonnes absentes",
                   "incohérences d'encaisse": int(grille["nb_incoherences_encaisse"].sum()) if AVEC_ENCAISSE else "–",
                   "référence": "{} → {}".format(DEBUT_REF.strftime("%d/%m"), (DEBUT - pd.Timedelta(days=1)).strftime("%d/%m")),
                   "période analysée": "{} → {}".format(DEBUT.strftime("%d/%m"), FIN.strftime("%d/%m")),
                   "date max des données": brut["ts"].max().strftime("%d/%m/%Y %Hh")},
                  "Données lues")
for message in ALERTES_DONNEES:
    afficher_titre("Attention : " + message)
if not ALERTES_DONNEES:
    afficher_titre("Contrôle des données : période de référence et période analysée bien couvertes.")

## 4. Comportement habituel de chaque terminal
Pour chaque position : **part habituelle** des billets sortis et **valeur habituelle** (la valeur déclarée la plus
fréquente, pondérée par les billets). Taux habituel de retraits répétés par la même carte.

In [ ]:
reference = grille[grille["ts"] < DEBUT]
habitude = pd.DataFrame(index=sorted(grille["num_automate"].unique()))
total_ref = reference.groupby("num_automate")[["nb_billets_pos{}".format(k) for k in POSITIONS]].sum()
for k in POSITIONS:
    habitude["part_ref_pos{}".format(k)] = total_ref["nb_billets_pos{}".format(k)] / total_ref.sum(axis=1).replace(0, np.nan)
    v = reference[reference["nb_billets_pos{}".format(k)] > 0].groupby(
        ["num_automate", "val_max_pos{}".format(k)])["nb_billets_pos{}".format(k)].sum().reset_index()
    v = v.sort_values("nb_billets_pos{}".format(k)).drop_duplicates("num_automate", keep="last")
    habitude["val_ref_pos{}".format(k)] = v.set_index("num_automate")["val_max_pos{}".format(k)]
somme_ref = reference.groupby("num_automate")[["nb_retraits", "nb_retraits_repetes_10min"]].sum()
habitude["taux_repetes_ref"] = (somme_ref["nb_retraits_repetes_10min"] / somme_ref["nb_retraits"].replace(0, np.nan))
habitude = habitude.fillna({"part_ref_pos{}".format(k): 0 for k in POSITIONS}).fillna({"taux_repetes_ref": 0})
for k in POSITIONS:
    habitude["active_pos{}".format(k)] = habitude["part_ref_pos{}".format(k)] >= PART_ACTIVE_MIN

# billets habituels par cassette, par jour de semaine et par heure (moyenne des 4 semaines de référence)
BILLETS = ["nb_billets_pos{}".format(k) for k in POSITIONS]
ATTENDUS = ["attendu_pos{}".format(k) for k in POSITIONS]
profil = reference.assign(js=reference["ts"].dt.dayofweek, hr=reference["ts"].dt.hour).groupby(
    ["num_automate", "js", "hr"])[BILLETS].mean()
profil.columns = ATTENDUS
grille = grille.assign(js=grille["ts"].dt.dayofweek, hr=grille["ts"].dt.hour).merge(
    profil.reset_index(), on=["num_automate", "js", "hr"], how="left").drop(columns=["js", "hr"])
grille[ATTENDUS] = grille[ATTENDUS].fillna(0)

recharges_ref = reference[reference["nb_recharges"] > 0]
habitude["chargements_par_semaine"] = (recharges_ref.groupby("num_automate").size() * 7 / JOURS_REFERENCE)
habitude["montant_charge_habituel"] = recharges_ref.groupby("num_automate")["montant_recharge"].median()
CHARGEMENTS = grille[grille["nb_recharges"] > 0][["num_automate", "ts", "montant_recharge"]]


def chargements_proches(terminal, t, marge_h=3):
    """Chargements du terminal à moins de marge_h heures de l'instant t (texte « 04/11 12h (31 000 €) »)."""
    c = CHARGEMENTS[(CHARGEMENTS["num_automate"] == terminal) &
                    ((CHARGEMENTS["ts"] - t).abs() <= pd.Timedelta(hours=marge_h))]
    return ", ".join("{} ({} €)".format(r.ts.strftime("%d/%m %Hh"), formater(r.montant_recharge)) for r in c.itertuples())


def plan_cassettes(terminal):
    """Texte « pos1 20 € (33 %) | pos2 ... » de l'habitude d'un terminal."""
    h = habitude.loc[terminal]
    return " | ".join("cassette {} : {} € ({:.0f} %{})".format(k, int(h["val_ref_pos{}".format(k)]) if not pd.isna(
        h["val_ref_pos{}".format(k)]) else "?", 100 * h["part_ref_pos{}".format(k)],
        "" if h["active_pos{}".format(k)] else ", inactive") for k in POSITIONS)


apercu = habitude.reset_index().rename(columns={"index": "num_automate"}).head(8)
apercu["plan habituel des cassettes"] = apercu["num_automate"].map(plan_cassettes)
afficher_tableau(apercu[["num_automate", "plan habituel des cassettes", "taux_repetes_ref",
                         "chargements_par_semaine", "montant_charge_habituel"]],
                 "Habitude des premiers terminal",
                 "Valeur habituelle de chaque position et part des billets qui en sortent ; taux habituel de "
                 "retraits répétés par la même carte en moins de 10 minutes ; chargements par semaine et montant "
                 "habituellement chargé (remontées de l'encaisse théorique).")

## 5. Calcul des signaux, heure par heure (technique)
Pour chaque terminal et chaque heure, on additionne les 24 dernières heures puis on calcule :
- **score_cassettes** : écart entre la répartition observée des billets et la répartition habituelle (statistique
  G du test du rapport de vraisemblance), divisé par le plus fort écart atteint par ce terminal en référence ;
- **ecart_part** : plus grand écart de part d'une position, en points ;
- **cassette sur-sollicitée / sous-utilisée** : billets sortis par une cassette active sur 24 h, divisés par ses
  billets habituels aux mêmes heures et mêmes jours de semaine (ratio) ;
- **cassette inactive réactivée** : une cassette inactive en référence se remet à sortir des billets ;
- **valeur_modifiee** : une position a sorti des billets avec une valeur déclarée différente de l'habitude ;
- **score_repetes** : excès de retraits répétés par rapport au taux habituel, divisé par le maximum de référence.

In [ ]:
def sommes_glissantes(table, colonnes, n):
    """Somme sur les n dernières heures, terminal par terminal (grille complète et triée)."""
    cumul = table.groupby("num_automate")[colonnes].cumsum()
    decale = cumul.groupby(table["num_automate"]).shift(n)
    return cumul - decale


grille = grille.sort_values(["num_automate", "ts"]).reset_index(drop=True)
h = habitude.reindex(grille["num_automate"]).reset_index(drop=True)

# valeur déclarée différente de l'habitude, heure par heure
modif = np.zeros(len(grille))
perte_param = np.zeros(len(grille))
for k in POSITIONS:
    ref_v = h["val_ref_pos{}".format(k)].values
    vmin, vmax = grille["val_min_pos{}".format(k)].values, grille["val_max_pos{}".format(k)].values
    billets = grille["nb_billets_pos{}".format(k)].values
    change = (billets > 0) & ~np.isnan(ref_v) & ((vmin != ref_v) | (vmax != ref_v)) & ~np.isnan(vmax)
    modif += change
    perte_param += np.where(change, billets * (ref_v - np.nan_to_num(vmax)), 0)
grille["heure_valeur_modifiee"] = (modif > 0).astype(int)
grille["perte_parametrage"] = perte_param

# billets par coupure habituelle (pour la perte en cas d'inversion 10 / 20)
grille["billets_pos_10"] = sum(np.where(h["val_ref_pos{}".format(k)].values == 10,
                                        grille["nb_billets_pos{}".format(k)].values, 0) for k in POSITIONS)
grille["billets_pos_20"] = sum(np.where(h["val_ref_pos{}".format(k)].values == 20,
                                        grille["nb_billets_pos{}".format(k)].values, 0) for k in POSITIONS)

A_SOMMER = (BILLETS + ATTENDUS +
            ["nb_retraits", "nb_retraits_repetes_10min", "heure_valeur_modifiee", "perte_parametrage",
             "billets_pos_10", "billets_pos_20"])
fen = sommes_glissantes(grille, A_SOMMER, FENETRE_HEURES)
fen["num_automate"], fen["ts"] = grille["num_automate"], grille["ts"]
fen = fen[fen["ts"] >= DEBUT_REF + pd.Timedelta(hours=FENETRE_HEURES - 1)].reset_index(drop=True)
hf = habitude.reindex(fen["num_automate"]).reset_index(drop=True)

# répartition entre positions : statistique G
ACTIF = hf[["active_pos{}".format(k) for k in POSITIONS]].values.astype(bool)
O = np.where(ACTIF, fen[BILLETS].values.astype(float), 0)          # cassettes actives seulement
B = O.sum(axis=1)
P = np.where(ACTIF, np.clip(hf[["part_ref_pos{}".format(k) for k in POSITIONS]].values, 1e-3, None), 0)
P = P / np.where(P.sum(axis=1, keepdims=True) > 0, P.sum(axis=1, keepdims=True), 1)
E = B[:, None] * P
with np.errstate(divide="ignore", invalid="ignore"):
    G = 2 * np.where(O > 0, O * np.log(O / E), 0).sum(axis=1)
parts = np.where(B[:, None] > 0, O / np.where(B[:, None] > 0, B[:, None], 1), np.nan)
ecarts = np.where(ACTIF, parts - P, 0)
fen["G"] = np.where(B >= MIN_BILLETS_FENETRE, G, np.nan)
fen["ecart_part"] = np.nanmax(np.abs(np.nan_to_num(ecarts)), axis=1)
fen["position_ecart"] = np.argmax(np.abs(np.nan_to_num(ecarts)), axis=1) + 1
fen["sens_ecart"] = np.sign(ecarts[np.arange(len(fen)), fen["position_ecart"].values - 1])
fen["nb_billets"] = B

# billets par cassette comparés à l'habitude des mêmes heures
OBS, ATT = fen[BILLETS].values.astype(float), fen[ATTENDUS].values.astype(float)
with np.errstate(divide="ignore", invalid="ignore"):
    RATIO = np.where(ATT > 0, OBS / ATT, np.nan)
SUR = ACTIF & (RATIO >= RATIO_SUR) & (OBS - ATT >= MIN_EXCES_BILLETS)
SOUS = ACTIF & (ATT >= MIN_BILLETS_HABITUELS) & (RATIO <= RATIO_SOUS)
REACT = ~ACTIF & (OBS >= MIN_BILLETS_REACTIVATION)
fen["signal_sur"], fen["signal_sous"], fen["signal_reactivation"] = SUR.any(1), SOUS.any(1), REACT.any(1)
# cassette la plus anormale en volume : écart au ratio 1 en échelle logarithmique
ecart_log = np.where(SUR | SOUS | REACT, np.abs(np.log(np.clip(np.where(np.isnan(RATIO), 10.0, RATIO), 0.02, 50))), -1)
fen["position_volume"] = np.argmax(ecart_log, axis=1) + 1
fen["billets_obs_pos"] = OBS[np.arange(len(fen)), fen["position_volume"].values - 1]
fen["billets_att_pos"] = ATT[np.arange(len(fen)), fen["position_volume"].values - 1]
fen["score_volume"] = np.where(ecart_log.max(axis=1) > 0, ecart_log.max(axis=1) / np.log(RATIO_SUR), 0)

# retraits répétés : écart au taux habituel, en écarts-types
r = hf["taux_repetes_ref"].values
n = fen["nb_retraits"].values
attendu = n * r
fen["z_repetes"] = (fen["nb_retraits_repetes_10min"].values - attendu) / np.sqrt(attendu * (1 - r) + 1)

# seuils propres à chaque terminal : maximum atteint pendant la référence
en_reference = fen["ts"] < DEBUT
max_ref = fen[en_reference].groupby("num_automate")[["G", "z_repetes"]].max()
plancher_G = chi2.ppf(0.9999, len(POSITIONS) - 1) if chi2 is not None else 20.0
seuil_G = np.maximum(max_ref["G"].reindex(fen["num_automate"]).values, plancher_G)
seuil_z = np.maximum(max_ref["z_repetes"].reindex(fen["num_automate"]).values, 4.0)
fen["score_cassettes"] = fen["G"] / seuil_G
fen["score_repetes"] = fen["z_repetes"] / seuil_z

fen["signal_cassettes"] = (fen["score_cassettes"] > MARGE_REFERENCE) & (fen["ecart_part"] >= ECART_PART_MIN)
fen["signal_valeur"] = fen["heure_valeur_modifiee"] > 0
fen["signal_repetes"] = (fen["score_repetes"] > MARGE_REFERENCE) & (fen["nb_retraits_repetes_10min"] >= MIN_REPETES)
fen["perte_si_inversion_10_20"] = 10 * (fen["billets_pos_10"] - fen["billets_pos_20"])
analyse = fen[fen["ts"] >= DEBUT + pd.Timedelta(hours=FENETRE_HEURES - 1)].copy()
SIGNAUX = [("répartition", "signal_cassettes"), ("cassette sur-sollicitée", "signal_sur"),
           ("cassette sous-utilisée", "signal_sous"), ("cassette inactive réactivée", "signal_reactivation"),
           ("valeur déclarée", "signal_valeur"), ("retraits répétés", "signal_repetes")]

afficher_chiffres({"fenêtres examinées": len(analyse),
                   "répartition anormale": int(analyse["signal_cassettes"].sum()),
                   "cassette sur-sollicitée": int(analyse["signal_sur"].sum()),
                   "cassette sous-utilisée": int(analyse["signal_sous"].sum()),
                   "cassette inactive réactivée": int(analyse["signal_reactivation"].sum()),
                   "valeur déclarée modifiée": int(analyse["signal_valeur"].sum()),
                   "retraits répétés": int(analyse["signal_repetes"].sum())},
                  "Fenêtres de 24 h en alerte sur la période analysée")

## 6. Anomalies de cassettes sur tout le parc
Les fenêtres en alerte qui se suivent forment un **événement**. Pour chaque événement, on retient la fenêtre de
24 h la plus anormale et on propose une lecture :
- **inversion 10 € / 20 € possible** : le terminal a des cassettes 10 € et 20 €, une de ces cassettes est
  sur-sollicitée ou sa répartition change, avec des retraits répétés ou un chargement juste avant ;
- **cassette 10 € ou 20 € sous-utilisée** : vide, bloquée, ou billets rejetés par le terminal après une inversion ;
- **cassette inactive réactivée** : une cassette inutilisée (ex. 5 €) ressort des billets : chargement à vérifier ;
- **erreur de paramétrage** : valeur déclarée modifiée puis revenue à l'habitude ;
- **nouveau plan de cassettes (à confirmer)** : valeur déclarée modifiée jusqu'à la fin des données ;
- **cassette vide ou bloquée** : une cassette sort au plus 30 % de ses billets habituels ;
- **retraits répétés** : clients qui refont un retrait, sans changement de répartition.

Chaque anomalie reçoit une **priorité** (haute, moyenne, basse) et une **phrase en clair**. Le tableau affiché
ne garde que l'essentiel ; mettre `AFFICHER_DETAIL = True` en section 1 pour voir toutes les colonnes.

In [ ]:
analyse["alerte"] = analyse[[cle for _, cle in SIGNAUX]].any(axis=1)
alertes = analyse[analyse["alerte"]].sort_values(["num_automate", "ts"]).copy()
alertes["nouvel_evenement"] = (alertes.groupby("num_automate")["ts"].diff() != pd.Timedelta(hours=1))
alertes["id_evenement"] = alertes["nouvel_evenement"].cumsum()
alertes["force"] = (alertes["score_cassettes"].fillna(0) + alertes["score_volume"].fillna(0)
                    + alertes["score_repetes"].clip(lower=0).fillna(0) + 2 * alertes["signal_valeur"])


def lire_evenement(groupe):
    """Résumé d'un événement : fenêtre la plus anormale, signaux, lecture proposée, pertes estimées."""
    pic = groupe.loc[groupe["force"].idxmax()]
    terminal = pic["num_automate"]
    hab = habitude.loc[terminal]
    en_volume = bool(pic["signal_sur"] or pic["signal_sous"] or pic["signal_reactivation"])
    pos = int(pic["position_volume"]) if en_volume else int(pic["position_ecart"])
    val_pos = hab["val_ref_pos{}".format(pos)]
    vals = set(hab[["val_ref_pos{}".format(k) for k in POSITIONS]].dropna().astype(int))
    part_obs = pic["nb_billets_pos{}".format(pos)] / max(pic["nb_billets"], 1)
    signaux = [nom for nom, cle in SIGNAUX if groupe[cle].any()]
    obs_pos = pic["nb_billets_pos{}".format(pos)]
    att_pos = pic["attendu_pos{}".format(pos)]
    dure_jusqua_fin = groupe["ts"].max() >= FIN - pd.Timedelta(hours=1)
    debut_pic = pic["ts"] - pd.Timedelta(hours=FENETRE_HEURES - 1)
    charge_debut = chargements_proches(terminal, debut_pic)
    charge_fin = chargements_proches(terminal, pic["ts"] + pd.Timedelta(hours=1))
    if groupe["signal_valeur"].any():
        lecture = "nouveau plan de cassettes (à confirmer)" if dure_jusqua_fin and len(groupe) > 48 \
            else "erreur de paramétrage de cassette"
    elif groupe["signal_reactivation"].any():
        lecture = "cassette inactive réactivée (chargement à vérifier)"
    elif ({10, 20} <= vals and val_pos in (10, 20) and (groupe["signal_sur"].any() or groupe["signal_cassettes"].any())
          and (groupe["signal_repetes"].any() or charge_debut or (val_pos == 10 and obs_pos > att_pos))):
        lecture = "inversion 10 € / 20 € possible"
    elif groupe["signal_sous"].any() and {10, 20} <= vals and val_pos in (10, 20):
        lecture = "cassette {} € sous-utilisée : vide, bloquée ou billets rejetés (inversion ?)".format(int(val_pos))
    elif groupe["signal_sous"].any():
        lecture = "cassette vide ou bloquée"
    elif groupe["signal_sur"].any():
        lecture = "cassette sur-sollicitée"
    elif groupe["signal_cassettes"].any():
        lecture = "répartition inhabituelle entre cassettes"
    else:
        lecture = "retraits répétés"
    valeur_txt = "{} €".format(int(val_pos)) if not pd.isna(val_pos) else "valeur inconnue"
    variation = "{:+.0f} %".format(100 * (obs_pos / att_pos - 1)) if att_pos > 0 else "habituellement aucun"
    explication = "Du {} au {}, la cassette {} (billets de {}) a distribué {} billets au lieu de {:.0f} " \
                  "habituellement ({}).".format(debut_pic.strftime("%d/%m %Hh"), pic["ts"].strftime("%d/%m %Hh"),
                                                pos, valeur_txt, int(obs_pos), att_pos, variation)
    if charge_debut:
        explication += " Le terminal a été chargé juste avant : {}.".format(charge_debut)
    if groupe["signal_repetes"].any():
        explication += " {} retraits refaits par la même carte en moins de 10 min (environ {:.0f} d'habitude).".format(
            int(pic["nb_retraits_repetes_10min"]), pic["nb_retraits"] * hab["taux_repetes_ref"])
    if groupe["signal_valeur"].any():
        explication += " La valeur déclarée d'une cassette a changé."
    explication += " Lecture : {}.".format(lecture)
    return pd.Series({
        "num_automate": terminal, "explication": explication,
        "site": infos.loc[terminal, "code_site_implantation"], "ville": infos.loc[terminal, "ville"],
        "lecture proposée": lecture, "signaux": ", ".join(signaux),
        "alerte du": groupe["ts"].min() - pd.Timedelta(hours=FENETRE_HEURES - 1), "au": groupe["ts"].max(),
        "24 h les plus anormales": "{} → {}".format(debut_pic.strftime("%d/%m %Hh"), pic["ts"].strftime("%d/%m %Hh")),
        "chargement au début": charge_debut or "non", "chargement à la fin": charge_fin or "non",
        "cassette en cause": "cassette {} ({} €)".format(pos, int(val_pos) if not pd.isna(val_pos) else "?"),
        "billets sortis (24 h)": int(obs_pos), "billets habituels": att_pos,
        "ratio": obs_pos / att_pos if att_pos > 0 else np.nan,
        "part habituelle %": 100 * hab["part_ref_pos{}".format(pos)], "part observée %": 100 * part_obs,
        "score cassettes": pic["score_cassettes"], "retraits répétés": int(pic["nb_retraits_repetes_10min"]),
        "attendus": pic["nb_retraits"] * hab["taux_repetes_ref"],
        "perte si inversion 10/20 (€)": pic["perte_si_inversion_10_20"] if {10, 20} <= vals else np.nan,
        "perte paramétrage (€)": pic["perte_parametrage"] if groupe["signal_valeur"].any() else np.nan,
        "force": pic["force"]})


def priorite(lecture):
    if lecture.startswith(("inversion", "erreur de paramétrage", "cassette inactive")) or "inversion ?" in lecture:
        return "1 - haute"
    if lecture.startswith(("cassette vide", "cassette sur", "nouveau plan")):
        return "2 - moyenne"
    return "3 - basse"


if len(alertes):
    evenements = alertes.groupby("id_evenement").apply(lire_evenement)
    evenements.insert(0, "priorité", evenements["lecture proposée"].map(priorite))
    evenements = evenements.sort_values(["priorité", "force"], ascending=[True, False]).reset_index(drop=True)
else:
    evenements = pd.DataFrame()
if len(evenements):
    compte = evenements["priorité"].value_counts()
    afficher_chiffres({"anomalies": len(evenements), "terminal concernés": evenements["num_automate"].nunique(),
                       "priorité haute": int(compte.get("1 - haute", 0)),
                       "priorité moyenne": int(compte.get("2 - moyenne", 0)),
                       "priorité basse": int(compte.get("3 - basse", 0))}, "Anomalies trouvées")
    ESSENTIEL = ["priorité", "num_automate", "site", "explication", "chargement au début",
                 "perte si inversion 10/20 (€)"]
    afficher_tableau(evenements[ESSENTIEL], "Les {} premières anomalies, de la plus urgente à la moins urgente".format(
        min(NB_ANOMALIES_AFFICHEES, len(evenements))),
        "Perte si inversion : estimation à n'utiliser que si l'inversion est confirmée par le métier.",
        max_lignes=NB_ANOMALIES_AFFICHEES)
else:
    afficher_titre("Aucune anomalie de cassette sur la période analysée.")
if AFFICHER_DETAIL and len(evenements):
    afficher_tableau(evenements.drop(columns=["force"]),
                 "Détail technique de toutes les anomalies ({} terminal)".format(
                     evenements["num_automate"].nunique() if len(evenements) else 0),
                 "Un terminal a 4 cassettes (bacs de billets), numérotées 1 à 4 ; la valeur entre parenthèses est la "
                 "coupure que la cassette contient d'habitude. Triés du plus fort au plus faible. Score cassettes : écart de répartition divisé par le plus fort "
                 "écart du même terminal en référence (au-dessus de 1, jamais vu en 4 semaines). Perte si inversion : "
                 "10 € x (billets des positions 10 € - billets des positions 20 €) sur les 24 h les plus anormales, "
                 "à n'utiliser que si l'inversion est confirmée.")

## 7. Vérification sur l'incident connu
Tous les terminaux dont le code regate (code_site_implantation) commence par le préfixe donné sont mesurés **exactement sur la période déclarée**, puis
comparés à leur habitude. Le terminal de l'incident doit ressortir nettement des autres, et sa perte estimée doit être
du même ordre que la perte déclarée. Un tableau jour par jour montre ensuite, pour chaque terminal et chaque cassette,
les billets sortis rapportés à l'habitude (1,0 = comme d'habitude ; 2,0 = deux fois plus).

In [ ]:
candidats = pd.DataFrame()
if not CAS_CONNU:
    afficher_titre("Section non exécutée : CAS_CONNU est vide (section 1)",
                   "Renseigner en section 1, par exemple : CAS_CONNU = {\"debut\": \"AAAA-MM-JJ HH:MM\", "
                   "\"fin\": \"AAAA-MM-JJ HH:MM\", \"prefixe_site\": \"123\", \"perte_declaree\": 5000, "
                   "\"jours_autour\": 2}, puis relancer les cellules à partir de la section 1.")
if CAS_CONNU:
    d0, d1 = pd.Timestamp(CAS_CONNU["debut"]), pd.Timestamp(CAS_CONNU["fin"])
    prefixe = str(CAS_CONNU["prefixe_site"]).strip().lstrip("0")

    def normaliser(s):
        return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")

    terminals = infos[normaliser(infos["code_site_implantation"]).str.startswith(prefixe) |
                 normaliser(infos["cd_regate"]).str.startswith(prefixe)].index
    if len(terminals) == 0:
        afficher_titre("Aucun terminal dont le code regate commence par {}".format(prefixe),
                       "Exemples de codes présents : {}".format(", ".join(infos["code_site_implantation"].astype(str)
                                                                         .drop_duplicates().head(15))))
    periode = grille[grille["num_automate"].isin(terminals) & (grille["ts"] >= d0) & (grille["ts"] < d1)]
    lignes = []
    for terminal, g in periode.groupby("num_automate"):
        hab = habitude.loc[terminal]
        o = np.array([g["nb_billets_pos{}".format(k)].sum() for k in POSITIONS], float)
        b = o.sum()
        p = np.clip(hab[["part_ref_pos{}".format(k) for k in POSITIONS]].values.astype(float), 1e-3, None)
        p = p / p.sum()
        with np.errstate(divide="ignore", invalid="ignore"):
            g_stat = 2 * np.where(o > 0, o * np.log(o / (b * p)), 0).sum() if b > 0 else np.nan
        seuil = max(max_ref["G"].get(terminal, np.nan) if terminal in max_ref.index else np.nan, plancher_G)
        ecart = o / max(b, 1) - hab[["part_ref_pos{}".format(k) for k in POSITIONS]].values
        k_max = int(np.argmax(np.abs(ecart)))
        vals = set(hab[["val_ref_pos{}".format(k) for k in POSITIONS]].dropna().astype(int))
        rep, n_ret = g["nb_retraits_repetes_10min"].sum(), g["nb_retraits"].sum()
        obs_k = np.array([g["nb_billets_pos{}".format(k)].sum() for k in POSITIONS], float)
        att_k = np.array([g["attendu_pos{}".format(k)].sum() for k in POSITIONS], float)
        act_k = np.array([bool(hab["active_pos{}".format(k)]) for k in POSITIONS])
        with np.errstate(divide="ignore", invalid="ignore"):
            r_k = np.where(act_k & (att_k > 0), obs_k / att_k, np.nan)
        k_sur = int(np.nanargmax(r_k)) if np.isfinite(r_k).any() else 0
        k_sous = int(np.nanargmin(r_k)) if np.isfinite(r_k).any() else 0
        inactives = [k for k in POSITIONS if not act_k[k - 1] and obs_k[k - 1] >= MIN_BILLETS_REACTIVATION]
        indice = np.nanmax([g_stat / seuil if b >= MIN_BILLETS_FENETRE else 0,
                            (r_k[k_sur] / RATIO_SUR) if np.isfinite(r_k[k_sur]) else 0,
                            (RATIO_SOUS / max(r_k[k_sous], 0.01)) if np.isfinite(r_k[k_sous]) and
                            att_k[k_sous] >= MIN_BILLETS_HABITUELS else 0, 2.0 if inactives else 0])
        lignes.append({
            "num_automate": terminal, "site": infos.loc[terminal, "code_site_implantation"], "ville": infos.loc[terminal, "ville"],
            "retraits": int(n_ret), "billets": int(b), "indice d'anomalie": indice,
            "cassette la plus sollicitée": "cassette {} ({} €) x{:.1f}".format(
                k_sur + 1, int(hab["val_ref_pos{}".format(k_sur + 1)]), r_k[k_sur]) if np.isfinite(r_k[k_sur]) else "–",
            "cassette la moins sollicitée": "cassette {} ({} €) x{:.1f}".format(
                k_sous + 1, int(hab["val_ref_pos{}".format(k_sous + 1)]), r_k[k_sous]) if np.isfinite(r_k[k_sous]) else "–",
            "cassette inactive réactivée": ", ".join("cassette {}".format(k) for k in inactives) or "non",
            "score cassettes": g_stat / seuil if b >= MIN_BILLETS_FENETRE else np.nan,
            "cassette la plus déviée": "cassette {} ({} €)".format(k_max + 1, int(hab["val_ref_pos{}".format(k_max + 1)]) if not pd.isna(hab["val_ref_pos{}".format(k_max + 1)]) else "?"),
            "part habituelle %": 100 * hab["part_ref_pos{}".format(k_max + 1)],
            "part observée %": 100 * o[k_max] / max(b, 1),
            "retraits répétés": int(rep), "attendus": n_ret * hab["taux_repetes_ref"],
            "valeur déclarée modifiée": bool(g["heure_valeur_modifiee"].sum() > 0),
            "chargement au début": chargements_proches(terminal, d0) or "non",
            "chargement à la fin": chargements_proches(terminal, d1) or "non",
            "incohérences d'encaisse": int(g["nb_incoherences_encaisse"].sum()),
            "perte si inversion 10/20 (€)": 10 * (g["billets_pos_10"].sum() - g["billets_pos_20"].sum())
            if {10, 20} <= vals else np.nan,
            "dans les événements": bool(len(evenements) and terminal in set(evenements["num_automate"]))})
    candidats = pd.DataFrame(lignes)
    if len(candidats):
        candidats["rang"] = candidats["indice d'anomalie"].rank(ascending=False, method="min")
        candidats = candidats.sort_values("indice d'anomalie", ascending=False).reset_index(drop=True)
    afficher_tableau(candidats, "terminal du site {}... du {} au {}".format(prefixe, d0.strftime("%d/%m %Hh"),
                                                                        d1.strftime("%d/%m %Hh")),
                     "Indice d'anomalie : le plus fort des signaux (répartition, cassette sur-sollicitée ou "
                     "sous-utilisée, cassette inactive réactivée) ; au-dessus de 1, jamais vu en référence. "
                     "x2,0 = la cassette a sorti deux fois ses billets habituels. Le terminal de l'incident est celui dont la perte estimée est proche de la perte déclarée ({} €) et "
                     "qui a été chargé au début de la période (l'inversion se fait au chargement)."
                     .format(formater(CAS_CONNU["perte_declaree"])))
    if len(candidats):
        premier = candidats.iloc[0]
        second = candidats["indice d'anomalie"].iloc[1] if len(candidats) > 1 else np.nan
        afficher_chiffres({"terminal le plus probable": premier["num_automate"],
                           "indice d'anomalie": premier["indice d'anomalie"],
                           "écart avec le 2e (x)": premier["indice d'anomalie"] / second if second and second > 0 else np.nan,
                           "perte estimée (€)": premier["perte si inversion 10/20 (€)"],
                           "perte déclarée (€)": CAS_CONNU["perte_declaree"]},
                          "Identification du cas connu")

        # tableau jour par jour, cassette par cassette (billets sortis / billets habituels)
        j0 = d0.floor("D") - pd.Timedelta(days=CAS_CONNU.get("jours_autour", 2))
        j1 = d1.floor("D") + pd.Timedelta(days=CAS_CONNU.get("jours_autour", 2) + 1)
        zone = grille[grille["num_automate"].isin(terminals) & (grille["ts"] >= j0) & (grille["ts"] < j1)].copy()
        JOURS_FR = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
        zone["jour"] = zone["ts"].dt.dayofweek.map(lambda j: JOURS_FR[j]) + " " + zone["ts"].dt.strftime("%d/%m")
        jours = zone.groupby(["num_automate", "jour"], sort=False)[BILLETS + ATTENDUS].sum()
        lignes_j = []
        for (terminal, jour), v in jours.iterrows():
            hab = habitude.loc[terminal]
            ligne = {"num_automate": terminal, "jour": jour}
            for k in POSITIONS:
                nom = "cassette {} ({} €)".format(k, int(hab["val_ref_pos{}".format(k)]) if not pd.isna(hab["val_ref_pos{}".format(k)]) else "?")
                o_, a_ = v["nb_billets_pos{}".format(k)], v["attendu_pos{}".format(k)]
                ligne[nom] = "{} / {:.0f} (x{:.1f})".format(int(o_), a_, o_ / a_) if a_ >= 1 else \
                    ("{} (inactive)".format(int(o_)) if not hab["active_pos{}".format(k)] else str(int(o_)))
            lignes_j.append(ligne)
        tableau_jours = pd.DataFrame(lignes_j)
        ordre = list(candidats["num_automate"])
        tableau_jours["ordre"] = tableau_jours["num_automate"].map({g: i for i, g in enumerate(ordre)})
        tableau_jours = tableau_jours.sort_values(["ordre"], kind="mergesort").drop(columns="ordre")
        for terminal in ordre[:5]:
            t = tableau_jours[tableau_jours["num_automate"] == terminal].dropna(axis=1, how="all")
            t = t.loc[:, [c for c in t.columns if c == "jour" or c.startswith("p")]]
            afficher_tableau(t, "{} — billets sortis / habituels, jour par jour".format(terminal),
                             "Chaque case : billets sortis / billets habituels du même jour de semaine (x ratio).")

## 8. Graphiques : billets sortis par coupure
Billets sortis par heure, regroupés par coupure **habituelle** de la position (10 €, 20 €, 50 €), du lundi au
vendredi autour de l'incident. Trait plein : observé ; pointillé : habitude du terminal à la même heure du même jour
de semaine (moyenne des 4 semaines de référence). Zone grisée : période de l'incident. Panneau du bas :
encaisse théorique du terminal ; traits verticaux : chargements (remontées de l'encaisse).

In [ ]:
def tracer_coupures(terminal, d0, d1, titre=None):
    hab = habitude.loc[terminal]
    g = grille[grille["num_automate"] == terminal].set_index("ts")
    zone = g[(g.index >= d0 - pd.Timedelta(days=2)) & (g.index <= d1 + pd.Timedelta(days=1))]
    ref = g[g.index < DEBUT]
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 4.6), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
    coupures = sorted(set(hab[["val_ref_pos{}".format(k) for k in POSITIONS]].dropna().astype(int)))
    for c in coupures:
        cols = ["nb_billets_pos{}".format(k) for k in POSITIONS if hab["val_ref_pos{}".format(k)] == c]
        obs = zone[cols].sum(axis=1)
        moy = ref[cols].sum(axis=1).groupby([ref.index.dayofweek, ref.index.hour]).mean()
        attendu = [moy.get((t.dayofweek, t.hour), np.nan) for t in zone.index]
        couleur = COULEUR_COUPURE.get(c, ENCRE)
        ax.plot(zone.index, obs, color=couleur, linewidth=2, label="{} € observé".format(c))
        ax.plot(zone.index, attendu, color=couleur, linewidth=1, linestyle="--", label="{} € habitude".format(c))
    ax.axvspan(d0, d1, color=GRIS_CLAIR, alpha=0.5, linewidth=0)
    ax2.axvspan(d0, d1, color=GRIS_CLAIR, alpha=0.5, linewidth=0)
    ax2.plot(zone.index, zone["encaisse_max_heure"].ffill(), color=GRIS, linewidth=1.5)
    ax2.set_ylabel("encaisse (€)")
    for t in zone.index[zone["nb_recharges"] > 0]:
        for a in (ax, ax2):
            a.axvline(t, color=ENCRE, linewidth=1, linestyle=":")
        ax2.annotate("chargement", (t, 1), xycoords=("data", "axes fraction"), xytext=(2, -9),
                     textcoords="offset points", fontsize=7, color=ENCRE)
    ax.set_ylabel("billets sortis par heure")
    ax.set_title(titre or "{} — billets sortis par coupure".format(terminal), pad=24)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%a %d/%m %Hh"))
    ax.legend(ncol=6, fontsize=8, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0))
    for e in ax2.get_xticklabels():
        e.set_fontsize(7)
    fig.tight_layout()
    plt.show()


a_tracer = []
if len(candidats):
    a_tracer.append((candidats.iloc[0]["num_automate"], pd.Timestamp(CAS_CONNU["debut"]),
                     pd.Timestamp(CAS_CONNU["fin"]), "cas connu — terminal le plus probable"))
    if len(candidats) > 1:
        a_tracer.append((candidats.iloc[1]["num_automate"], pd.Timestamp(CAS_CONNU["debut"]),
                         pd.Timestamp(CAS_CONNU["fin"]), "cas connu — 2e terminal du site (comparaison)"))
for _, ev in evenements.head(4).iterrows() if len(evenements) else []:
    if ev["num_automate"] not in [a[0] for a in a_tracer]:
        a_tracer.append((ev["num_automate"], ev["alerte du"], ev["au"], ev["lecture proposée"]))
for terminal, d0, d1, libelle in a_tracer:
    tracer_coupures(terminal, d0, d1, "{} — {} ({})".format(terminal, libelle, infos.loc[terminal, "ville"]))

## 9. Résumé et export

In [ ]:
afficher_chiffres({"terminal analysés": grille["num_automate"].nunique(),
                   "période analysée": "{} → {}".format(DEBUT.strftime("%d/%m"), FIN.strftime("%d/%m %Hh")),
                   "anomalies": len(evenements),
                   "inversions 10/20 possibles": int(evenements["lecture proposée"].str.startswith("inversion").sum())
                   if len(evenements) else 0,
                   "erreurs de paramétrage": int(evenements["lecture proposée"].str.startswith("erreur").sum())
                   if len(evenements) else 0,
                   "cassettes vides ou bloquées": int(evenements["lecture proposée"].str.contains("vide").sum())
                   if len(evenements) else 0},
                  "Résumé")
if len(evenements):
    afficher_titre("Les 10 anomalies les plus urgentes")
    for _, ev in evenements.head(10).iterrows():
        texte = "<b>[{}] {}</b> : {}".format(ev["priorité"][4:], ev["num_automate"], echapper(ev["explication"]))
        if not pd.isna(ev["perte si inversion 10/20 (€)"]) and ev["lecture proposée"].startswith("inversion"):
            texte += " ; perte estimée si l'inversion est confirmée : {} €".format(formater(ev["perte si inversion 10/20 (€)"]))
        display(HTML(CSS + "<div class='note'>{}</div>".format(texte)))
if DATASET_SORTIE_EVENEMENTS and len(evenements):
    import dataiku
    sortie = evenements.copy()
    for c in ["alerte du", "au"]:
        sortie[c] = sortie[c].astype(str)
    dataiku.Dataset(DATASET_SORTIE_EVENEMENTS).write_with_schema(sortie)

## 10. Zoom sur les terminaux et la période de l'incident connu
Uniquement les terminaux dont le `code_site_implantation` commence par le préfixe de la section 1, et uniquement les
alertes qui touchent la période donnée (élargie de `jours_autour` jours de chaque côté). La date maximale des
données indique si la période est bien couverte.

In [ ]:
if not CAS_CONNU:
    afficher_titre("Section non exécutée : CAS_CONNU est vide (section 1)",
                   "Renseigner en section 1, par exemple : CAS_CONNU = {\"debut\": \"AAAA-MM-JJ HH:MM\", "
                   "\"fin\": \"AAAA-MM-JJ HH:MM\", \"prefixe_site\": \"123\", \"perte_declaree\": 5000, "
                   "\"jours_autour\": 2}, puis relancer les cellules à partir de la section 1.")
if CAS_CONNU:
    z0, z1 = pd.Timestamp(CAS_CONNU["debut"]), pd.Timestamp(CAS_CONNU["fin"])
    marge = pd.Timedelta(days=CAS_CONNU.get("jours_autour", 2))
    pref = str(CAS_CONNU["prefixe_site"]).strip().lstrip("0")
    codes = infos["code_site_implantation"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
    terminal_zoom = codes[codes.str.startswith(pref)].index
    derniere = brut[brut["num_automate"].isin(terminal_zoom)].groupby("num_automate")["ts"].max()
    afficher_chiffres({"code regate commençant par": pref,
                       "terminal concernés": len(terminal_zoom),
                       "période donnée": "{} → {}".format(z0.strftime("%d/%m %Hh"), z1.strftime("%d/%m %Hh")),
                       "date max des données (tous terminal)": brut["ts"].max().strftime("%d/%m/%Y %Hh"),
                       "date max des données (ces terminaux)": derniere.max().strftime("%d/%m/%Y %Hh") if len(derniere) else "–",
                       "période couverte": "oui" if len(derniere) and derniere.max() >= z1 - pd.Timedelta(hours=1)
                       else "NON : données arrêtées avant la fin de la période"},
                      "Zoom code regate {} du {} au {}".format(pref, z0.strftime("%d/%m"), z1.strftime("%d/%m")))

    # dernière date de chaque terminal : un terminal qui s'arrête avant la période ne peut pas être analysé
    table_dates = pd.DataFrame({"num_automate": derniere.index,
                                "dernière heure présente": derniere.values}).sort_values("dernière heure présente")
    afficher_tableau(table_dates, "Dernière heure présente dans les données, par terminal")

    # événements de la section 6 qui touchent la période
    if len(evenements):
        ev_zoom = evenements[evenements["num_automate"].isin(terminal_zoom) &
                             (evenements["au"] >= z0 - marge) & (evenements["alerte du"] <= z1 + marge)]
    else:
        ev_zoom = pd.DataFrame()
    if len(ev_zoom):
        afficher_titre("En clair : ce qui s'est passé sur ces terminaux autour de la période",
                       "Un terminal a 4 cassettes (bacs de billets) numérotées 1 à 4. Du plus fort au plus faible.")
        for _, ev in ev_zoom.iterrows():
            display(HTML(CSS + "<div class='note'><b>{}</b> : {}</div>".format(ev["num_automate"],
                                                                             echapper(ev["explication"]))))
    if not len(ev_zoom):
        afficher_titre("Aucune anomalie sur ces terminaux autour de la période.")
    if AFFICHER_DETAIL and len(ev_zoom):
        afficher_tableau(ev_zoom.drop(columns=["force"]) if len(ev_zoom) else ev_zoom,
                     "Détail des événements de ces terminaux autour de la période ({} événement(s))".format(len(ev_zoom)),
                     "Événements de la section 6 dont les dates chevauchent la période donnée, élargie de {} jours."
                     .format(marge.days))

    # fenêtres de 24 h en alerte, heure par heure, pour ces terminaux et cette période
    fen_zoom = analyse[analyse["num_automate"].isin(terminal_zoom) & (analyse["ts"] >= z0 - marge) &
                       (analyse["ts"] <= z1 + marge) & analyse["alerte"]].copy()
    if len(fen_zoom):
        fen_zoom["signaux"] = fen_zoom.apply(lambda l: ", ".join(nom for nom, cle in SIGNAUX if l[cle]), axis=1)
        fen_zoom["fenêtre de 24 h"] = fen_zoom["ts"].apply(
            lambda t: "{} → {}".format((t - pd.Timedelta(hours=FENETRE_HEURES - 1)).strftime("%d/%m %Hh"),
                                       t.strftime("%d/%m %Hh")))
        en_vol = fen_zoom[["signal_sur", "signal_sous", "signal_reactivation"]].any(axis=1)
        fen_zoom["k"] = np.where(en_vol, fen_zoom["position_volume"], fen_zoom["position_ecart"]).astype(int)
        fen_zoom["cassette en cause"] = fen_zoom.apply(lambda l: "cassette {} ({} €)".format(
            l["k"], formater(habitude.loc[l["num_automate"], "val_ref_pos{}".format(l["k"])])).replace(".0 €", " €"),
            axis=1)
        fen_zoom["billets sortis"] = fen_zoom.apply(lambda l: l["nb_billets_pos{}".format(l["k"])], axis=1)
        fen_zoom["billets habituels"] = fen_zoom.apply(lambda l: l["attendu_pos{}".format(l["k"])], axis=1)
        fen_zoom = fen_zoom[["num_automate", "fenêtre de 24 h", "signaux", "cassette en cause", "billets sortis",
                             "billets habituels", "score_cassettes", "score_volume", "nb_retraits_repetes_10min"]]
    afficher_tableau(fen_zoom, "Fenêtres de 24 h en alerte pour ces terminaux ({} fenêtre(s))".format(len(fen_zoom)),
                     "Une ligne par heure de fin de fenêtre ; billets sortis et habituels de la cassette en cause "
                     "sur les 24 h de la fenêtre.", max_lignes=60)

## 11. Zoom sur 1 à 3 terminaux au choix : l'étude d'inversion refaite, avec son impact
Renseigner `ZOOM_TERMINAL` en section 1 (1 à 3 numéros d'automate ou codes site), puis relancer à partir de la
section 1. Laissé vide, le zoom prend le terminal le plus probable du cas connu (le cas du 95) et les inversions
possibles de la section 6.

Pour chaque terminal, l'étude d'inversion est refaite sur la **période étudiée** (zone grisée) : la période de
`ZOOM_PERIODE_SUSPECTE`, sinon celle du cas connu, sinon les 24 h où la part des billets de 10 € s'écarte le plus
de l'habitude.

**Bandeaux d'impact**, datés, puis détaillés **jour par jour** (affichés quand l'inversion est déclarée ou que le profil y ressemble), en supposant que
les cassettes 10 € et 20 € ont été échangées pendant la période étudiée :
- **retraits concernés** : retraits qui ont puisé au moins un billet dans une cassette déclarée 10 € (le client a
  reçu un billet de 20 € à la place) ;
- **distribués en trop** : 10 € x billets sortis des cassettes déclarées 10 € ;
- **impact 20 €** : retraits qui ont puisé dans une cassette déclarée 20 € (le client a tiré un billet de 20 € et
  reçu un billet de 10 €) et **distribués en moins** : 10 € x billets sortis de ces cassettes ;
- **perte nette** : distribués en trop - distribués en moins, à comparer à la perte déclarée.

Trois graphiques par terminal, heure par heure : billets sortis par coupure face à l'habitude (pointillé) ; part des
billets de 10 € sur 24 h glissantes face à la part habituelle ; retraits refaits par la même carte en moins de
10 min. Traits verticaux : chargements. Avec 2 ou 3 terminaux, un dernier graphique les superpose.

In [ ]:
CSS_IMPACT = """<style>
.imp{display:flex;align-items:center;gap:14px;border:2px solid #e0457b;border-radius:10px;background:#fff6f8;
     padding:9px 14px;margin:6px 0;max-width:1000px}
.imp .pill{background:#e0457b;color:#fff;font-weight:700;font-size:11px;letter-spacing:1.5px;border-radius:14px;
     padding:4px 14px;white-space:nowrap}
.imp .txt{color:#3b3fb6;font-weight:700;font-size:15px}
.imp .nt{color:#6b6a66;font-size:12px;font-weight:400;margin-left:6px}
.imp.gris{border-color:#b5b4ae;background:#fafaf8}.imp.gris .pill{background:#8a8984}
</style>"""


def bandeau_impact(etiquette, texte, note="", gris=False):
    """Bandeau « ÉTIQUETTE  texte (note) »."""
    display(HTML(CSS_IMPACT + "<div class='imp{}'><span class='pill'>{}</span><span class='txt'>{}"
                 "<span class='nt'>{}</span></span></div>".format(" gris" if gris else "", echapper(etiquette.upper()),
                                                                 echapper(texte), echapper(note))))


def _code(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


demandes = [str(g).strip() for g in (ZOOM_TERMINAL or []) if str(g).strip()]
terminal_zoom_choisis, introuvables = [], []
for g in demandes:
    if g in habitude.index:
        terminal_zoom_choisis.append(g)
        continue
    trouves = infos.index[(_code(infos["code_site_implantation"]) == g) | (_code(infos["cd_regate"]) == g)]
    if len(trouves):
        terminal_zoom_choisis.extend(trouves)
    else:
        introuvables.append(g)
if not demandes:
    if len(candidats):
        terminal_zoom_choisis.append(candidats.iloc[0]["num_automate"])
    if len(evenements):
        terminal_zoom_choisis.extend(evenements.loc[evenements["lecture proposée"].str.startswith("inversion"), "num_automate"])
terminal_zoom_choisis = list(dict.fromkeys(terminal_zoom_choisis))[:3]

# période étudiée (zone grisée) et période affichée
if ZOOM_PERIODE_SUSPECTE:
    s0, s1 = pd.Timestamp(ZOOM_PERIODE_SUSPECTE[0]), pd.Timestamp(ZOOM_PERIODE_SUSPECTE[1])
elif CAS_CONNU:
    s0, s1 = pd.Timestamp(CAS_CONNU["debut"]), pd.Timestamp(CAS_CONNU["fin"])
else:
    s0 = s1 = None


def _fin_de_jour(texte):
    t = pd.Timestamp(texte)
    return t + pd.Timedelta(hours=23) if len(str(texte).strip()) <= 10 else t


zd0 = pd.Timestamp(ZOOM_DEBUT) if ZOOM_DEBUT else (s0.floor("D") - pd.Timedelta(days=2) if s0 is not None
                                                     else FIN.floor("D") - pd.Timedelta(days=7))
zd1 = _fin_de_jour(ZOOM_FIN) if ZOOM_FIN else (s1.floor("D") + pd.Timedelta(days=3, hours=-1) if s1 is not None else FIN)
zd0, zd1 = max(zd0, grille["ts"].min()), min(zd1, FIN)

if introuvables:
    afficher_titre("Attention : terminal introuvable(s) dans les données : {}".format(", ".join(introuvables)),
                   "Vérifier le numéro d'automate (num_automate) ou le code site ; exemples présents : {}.".format(
                       ", ".join(list(habitude.index[:8]))))
if not terminal_zoom_choisis:
    afficher_titre("Section non exécutée : aucun terminal à zoomer",
                   "Renseigner en section 1, par exemple : ZOOM_TERMINAL = [\"123921\"] (1 à 3 terminaux), puis relancer.")
elif zd0 >= zd1:
    afficher_titre("Section non exécutée : la période du zoom ({} → {}) n'est pas dans les données ({} → {})".format(
        zd0.strftime("%d/%m/%Y"), zd1.strftime("%d/%m/%Y"), grille["ts"].min().strftime("%d/%m/%Y"),
        FIN.strftime("%d/%m/%Y")))
    terminal_zoom_choisis = []

sous_grille = grille[grille["num_automate"].isin(terminal_zoom_choisis)]
ecarts_zoom, recap_zoom = {}, []


def etude_inversion(terminal):
    """Refait l'étude d'inversion 10 / 20 sur un terminal : bandeaux d'impact, graphiques heure par heure, synthèse."""
    hab = habitude.loc[terminal]
    g = sous_grille[sous_grille["num_automate"] == terminal].set_index("ts").sort_index()
    val = {k: hab["val_ref_pos{}".format(k)] for k in POSITIONS}
    p10 = [k for k in POSITIONS if val[k] == 10 and hab["active_pos{}".format(k)]]
    p20 = [k for k in POSITIONS if val[k] == 20 and hab["active_pos{}".format(k)]]
    b10 = g[["nb_billets_pos{}".format(k) for k in p10]].sum(axis=1)
    b20 = g[["nb_billets_pos{}".format(k) for k in p20]].sum(axis=1)
    a10 = g[["attendu_pos{}".format(k) for k in p10]].sum(axis=1)
    a20 = g[["attendu_pos{}".format(k) for k in p20]].sum(axis=1)
    avec_10_20 = bool(p10) and bool(p20)
    r10, r20 = hab[["part_ref_pos{}".format(k) for k in p10]].sum(), hab[["part_ref_pos{}".format(k) for k in p20]].sum()
    part_hab = r10 / (r10 + r20) if avec_10_20 and r10 + r20 > 0 else np.nan
    s10 = b10.rolling(FENETRE_HEURES, min_periods=FENETRE_HEURES).sum()
    s20 = b20.rolling(FENETRE_HEURES, min_periods=FENETRE_HEURES).sum()
    part_24h = (s10 / (s10 + s20).where(s10 + s20 >= MIN_BILLETS_FENETRE)) if avec_10_20 else pd.Series(np.nan, index=g.index)
    zone = g.loc[zd0:zd1]
    pz = part_24h.loc[zd0:zd1]

    # période étudiée : période suspecte si elle touche le zoom, sinon les 24 h les plus écartées de l'habitude
    if s0 is not None and s1 > zd0 and s0 < zd1:
        e0, e1, origine = max(s0, zd0), min(s1, zd1 + pd.Timedelta(hours=1)), "période suspecte"
    elif pz.notna().any():
        t_max = (pz - part_hab).abs().idxmax()
        e0, e1, origine = t_max - pd.Timedelta(hours=FENETRE_HEURES - 1), t_max + pd.Timedelta(hours=1), "24 h les plus écartées"
    else:
        e0, e1, origine = zd0, zd1 + pd.Timedelta(hours=1), "période affichée"
    etu = g[(g.index >= e0) & (g.index < e1)]
    o10, o20 = b10.loc[etu.index].sum(), b20.loc[etu.index].sum()
    h10, h20 = a10.loc[etu.index].sum(), a20.loc[etu.index].sum()
    n10 = etu[["nb_retraits_pos{}".format(k) for k in p10]].sum().sum()     # retraits qui ont puisé dans une cassette 10 €
    n20 = etu[["nb_retraits_pos{}".format(k) for k in p20]].sum().sum()
    ret_par_pos = n10 + n20 > 0 or o10 + o20 == 0                            # colonnes nb_retraits_pos présentes
    part_obs = o10 / (o10 + o20) if avec_10_20 and o10 + o20 > 0 else np.nan
    rep, rep_att = etu["nb_retraits_repetes_10min"].sum(), etu["nb_retraits"].sum() * hab["taux_repetes_ref"]
    charges = zone.index[zone["nb_recharges"] > 0]
    charge_debut, charge_fin = chargements_proches(terminal, e0), chargements_proches(terminal, e1)
    ecart_pts = 100 * (part_obs - part_hab) if pd.notnull(part_obs) and pd.notnull(part_hab) else np.nan
    trop, moins = 10 * o10, 10 * o20
    duree_h = int(round((e1 - e0) / pd.Timedelta(hours=1)))

    compatible = False
    if not avec_10_20:
        verdict = "pas d'inversion 10 / 20 possible : le terminal n'a pas à la fois une cassette 10 € et une cassette 20 € actives"
    elif pd.notnull(ecart_pts) and abs(ecart_pts) >= 100 * ECART_PART_MIN and (charge_debut or rep > max(MIN_REPETES, 2 * rep_att)):
        compatible = True
        verdict = "profil compatible avec une inversion 10 € / 20 € (part des 10 € {:+.0f} points{}{})".format(
            ecart_pts, ", chargement au début" if charge_debut else "",
            ", retraits refaits en excès" if rep > max(MIN_REPETES, 2 * rep_att) else "")
    elif pd.notnull(ecart_pts) and abs(ecart_pts) >= 100 * ECART_PART_MIN:
        verdict = "répartition 10 € / 20 € inhabituelle ({:+.0f} points) sans chargement au début : à vérifier".format(ecart_pts)
    else:
        verdict = "pas de trace d'inversion : répartition 10 € / 20 € conforme à l'habitude"

    # bandeaux d'impact
    afficher_titre("{} — site {}{}".format(terminal, infos.loc[terminal, "code_site_implantation"],
                                         " ({})".format(infos.loc[terminal, "ville"]) if str(infos.loc[terminal, "ville"]).strip() else ""),
                   "{} du {} au {} : {}.".format(origine.capitalize(), e0.strftime("%d/%m %Hh"), e1.strftime("%d/%m %Hh"),
                                                verdict))
    terminal_du_cas = bool(len(candidats)) and terminal == candidats.iloc[0]["num_automate"] and origine == "période suspecte"
    jfr_ = lambda t: ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"][t.dayofweek] + t.strftime(" %d/%m/%Y %Hh")
    jour_txt = lambda t: ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"][t.dayofweek] + t.strftime(" %d/%m/%Y")
    quand = "du {} au {}".format(jfr_(e0), jfr_(e1))
    if avec_10_20 and (compatible or terminal_du_cas):
        bandeau_impact("date", "événement {}".format(quand),
                       "{} h ; chargement au début : {} ; à la fin : {}".format(duree_h, charge_debut or "non", charge_fin or "non"))
        bandeau_impact("impact 10 €", "{} retraits concernés · {} € distribués en trop".format(
            formater(int(n10)) if ret_par_pos else "?", formater(int(trop))),
            "{} ; clients qui ont tiré un billet de 10 € et reçu un billet de 20 € ({} billets)".format(quand, formater(int(o10))))
        bandeau_impact("impact 20 €", "{} retraits concernés · {} € distribués en moins".format(
            formater(int(n20)) if ret_par_pos else "?", formater(int(moins))),
            "{} ; clients qui ont tiré un billet de 20 € et reçu un billet de 10 € ({} billets) : "
            "ils ont été lésés".format(quand, formater(int(o20))))
        perte_decl = CAS_CONNU.get("perte_declaree") if CAS_CONNU and origine == "période suspecte" else None
        bandeau_impact("perte nette", "{} € de perte estimée pour la banque".format(formater(int(trop - moins))),
                       "{} € en trop - {} € en moins{}".format(formater(int(trop)), formater(int(moins)),
                                                               " ; perte déclarée : {} €".format(formater(perte_decl)) if perte_decl else ""))
        bandeau_impact("clients", "{} retraits refaits en moins de 10 min".format(formater(int(rep))),
                       "{} ; environ {:.0f} d'habitude sur la même durée".format(quand, rep_att))
        bandeau_impact("signal", "part des billets de 10 € : {:.0f} % au lieu de {:.0f} %".format(100 * part_obs, 100 * part_hab)
                       if pd.notnull(part_obs) else "part des billets de 10 € : –",
                       "billets 10 € : {} sortis pour {:.0f} habituels ; billets 20 € : {} pour {:.0f}".format(
                           formater(int(o10)), h10, formater(int(o20)), h20))

        # détail jour par jour : quel jour l'événement a pesé
        c10 = ["nb_retraits_pos{}".format(k) for k in p10]
        c20 = ["nb_retraits_pos{}".format(k) for k in p20]
        par_jour = pd.DataFrame({"r10": etu[c10].sum(axis=1), "r20": etu[c20].sum(axis=1),
                                 "b10": b10.loc[etu.index], "b20": b20.loc[etu.index],
                                 "rep": etu["nb_retraits_repetes_10min"]})
        par_jour["heure"] = par_jour.index
        lignes_j = []
        for d, v in par_jour.groupby(par_jour.index.floor("D")):
            lignes_j.append({"jour": jour_txt(d),
                             "heures": "{}h à {}h".format(v["heure"].min().hour, v["heure"].max().hour + 1),
                             "retraits billet 10 €": int(v["r10"].sum()) if ret_par_pos else np.nan,
                             "distribués en trop (€)": int(10 * v["b10"].sum()),
                             "retraits billet 20 €": int(v["r20"].sum()) if ret_par_pos else np.nan,
                             "distribués en moins (€)": int(10 * v["b20"].sum()),
                             "perte nette (€)": int(10 * (v["b10"].sum() - v["b20"].sum())),
                             "retraits refaits": int(v["rep"].sum())})
        for l in lignes_j:
            bandeau_impact(l["jour"].split(" ")[0][:3] + " " + l["jour"].split(" ")[1][:5],
                           "{} retraits 10 € · {} € en trop  |  {} retraits 20 € · {} € en moins  |  perte nette {} €".format(
                               formater(l["retraits billet 10 €"]) if ret_par_pos else "?", formater(l["distribués en trop (€)"]),
                               formater(l["retraits billet 20 €"]) if ret_par_pos else "?", formater(l["distribués en moins (€)"]),
                               formater(l["perte nette (€)"])),
                           "{}, de {}".format(l["jour"], l["heures"]))
        afficher_tableau(pd.DataFrame(lignes_j), "{} — impact jour par jour".format(terminal),
                         "Retraits billet 10 € : retraits qui ont puisé dans la cassette déclarée 10 € (le client a "
                         "reçu des billets de 20 €). Retraits billet 20 € : retraits qui ont puisé dans la cassette "
                         "déclarée 20 € (le client a reçu des billets de 10 €).")
    else:
        bandeau_impact("impact", "aucun impact d'inversion sur ce terminal", verdict + (
            " ; pour comparaison, perte nette si inversion : {} €".format(formater(int(trop - moins))) if avec_10_20 else ""),
            gris=True)

    # graphiques
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 7.2), sharex=True, gridspec_kw={"height_ratios": [3, 2, 1.4]})
    for c in sorted(set(int(v) for k, v in val.items() if pd.notnull(v) and hab["active_pos{}".format(k)])):
        ks = [k for k in POSITIONS if val[k] == c and hab["active_pos{}".format(k)]]
        couleur = COULEUR_COUPURE.get(c, ENCRE)
        ax1.plot(zone.index, zone[["nb_billets_pos{}".format(k) for k in ks]].sum(axis=1), color=couleur, linewidth=2,
                 label="{} € sortis".format(c))
        ax1.plot(zone.index, zone[["attendu_pos{}".format(k) for k in ks]].sum(axis=1), color=couleur, linewidth=1,
                 linestyle="--", label="{} € habitude".format(c))
    ax1.set_ylabel("billets par heure")
    ax1.legend(ncol=6, fontsize=8, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0))
    ax1.set_title("{} : {}".format(terminal, plan_cassettes(terminal)), pad=24, fontsize=10)
    if avec_10_20:
        ax2.plot(pz.index, 100 * pz, color=ORANGE, linewidth=2, label="part des 10 € sur 24 h")
        ax2.axhline(100 * part_hab, color=GRIS, linewidth=1, linestyle="--", label="part habituelle")
        ax2.axhspan(100 * (part_hab - ECART_PART_MIN), 100 * (part_hab + ECART_PART_MIN), color=GRIS_CLAIR, alpha=0.35,
                    linewidth=0, label="habitude ± {:.0f} points".format(100 * ECART_PART_MIN))
        ax2.set_ylabel("% de billets de 10 €")
        ax2.legend(ncol=3, fontsize=8, frameon=False, loc="upper left")
    else:
        ax2.text(0.5, 0.5, "pas de cassettes 10 € et 20 € actives sur ce terminal", transform=ax2.transAxes,
                 ha="center", va="center", color=GRIS)
    ax3.bar(zone.index, zone["nb_retraits_repetes_10min"], width=1 / 24 * 0.8, color=ENCRE, label="retraits refaits")
    ax3.plot(zone.index, zone["nb_retraits"] * hab["taux_repetes_ref"], color=GRIS, linewidth=1, linestyle="--",
             label="attendus")
    ax3.set_ylabel("refaits < 10 min")
    ax3.legend(ncol=2, fontsize=8, frameon=False, loc="upper left")
    for a in (ax1, ax2, ax3):
        a.axvspan(e0, e1, color=GRIS_CLAIR, alpha=0.5, linewidth=0, zorder=0)
        for t in charges:
            a.axvline(t, color=ENCRE, linewidth=1, linestyle=":")
    for t in charges:
        ax1.annotate("chargement", (t, 1), xycoords=("data", "axes fraction"), xytext=(2, -9),
                     textcoords="offset points", fontsize=7, color=ENCRE)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%a %d/%m %Hh"))
    for e in ax3.get_xticklabels():
        e.set_fontsize(7)
    fig.tight_layout()
    plt.show()

    if avec_10_20:
        ecarts_zoom[terminal] = 100 * (pz - part_hab)
    ev = evenements[(evenements["num_automate"] == terminal) & (evenements["au"] >= zd0) &
                    (evenements["alerte du"] <= zd1)] if len(evenements) else pd.DataFrame()
    for _, e in ev.iterrows():
        display(HTML(CSS + "<div class='note'>Section 6 : {}</div>".format(echapper(e["explication"]))))
    return {"num_automate": terminal, "site": infos.loc[terminal, "code_site_implantation"], "ville": infos.loc[terminal, "ville"],
            "date de l'événement": "{} → {}".format(e0.strftime("%d/%m/%Y %Hh"), e1.strftime("%d/%m/%Y %Hh")),
            "retraits concernés (10 €)": int(n10) if ret_par_pos else np.nan,
            "distribués en trop (€)": int(trop) if avec_10_20 else np.nan,
            "retraits concernés (20 €)": int(n20) if ret_par_pos else np.nan,
            "distribués en moins (€)": int(moins) if avec_10_20 else np.nan,
            "perte nette si inversion (€)": int(trop - moins) if avec_10_20 else np.nan,
            "billets 10 € sortis / habituels": "{} / {:.0f} (x{:.1f})".format(int(o10), h10, o10 / h10) if h10 >= 1 else "–",
            "billets 20 € sortis / habituels": "{} / {:.0f} (x{:.1f})".format(int(o20), h20, o20 / h20) if h20 >= 1 else "–",
            "part 10 € habituelle %": 100 * part_hab, "part 10 € observée %": 100 * part_obs,
            "retraits refaits": int(rep), "attendus": rep_att,
            "chargement au début": charge_debut or "non", "chargement à la fin": charge_fin or "non",
            "anomalies section 6": len(ev), "lecture": verdict}


if terminal_zoom_choisis:
    afficher_chiffres({"terminal zoomés": ", ".join(terminal_zoom_choisis),
                       "choisis": "par toi (ZOOM_TERMINAL)" if demandes else "automatiquement (cas connu, inversions)",
                       "période affichée": "{} → {}".format(zd0.strftime("%d/%m"), zd1.strftime("%d/%m")),
                       "période suspecte": "{} → {}".format(s0.strftime("%d/%m %Hh"), s1.strftime("%d/%m %Hh"))
                       if s0 is not None else "aucune (24 h les plus écartées)"},
                      "Zoom sur {} terminal".format(len(terminal_zoom_choisis)))
    for terminal in terminal_zoom_choisis:
        recap_zoom.append(etude_inversion(terminal))
    recap_zoom = pd.DataFrame(recap_zoom)
    afficher_tableau(recap_zoom, "Synthèse de l'étude d'inversion sur les terminaux zoomés",
                     "Montants calculés en supposant l'inversion des cassettes 10 € et 20 € sur la période étudiée : "
                     "à n'utiliser que si l'inversion est confirmée par le métier. Une perte nette négative ou faible "
                     "veut dire que la répartition ne ressemble pas à une inversion.")

if len(ecarts_zoom) >= 2:
    COULEURS_TERMINAL = [ENCRE, "#7b5ea7", "#1a9e77"]
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.axhspan(-100 * ECART_PART_MIN, 100 * ECART_PART_MIN, color=GRIS_CLAIR, alpha=0.35, linewidth=0,
               label="habitude ± {:.0f} points".format(100 * ECART_PART_MIN))
    ax.axhline(0, color=GRIS, linewidth=1)
    if s0 is not None and s1 > zd0 and s0 < zd1:
        ax.axvspan(max(s0, zd0), min(s1, zd1), color=GRIS_CLAIR, alpha=0.5, linewidth=0, zorder=0)
    for i, (terminal, serie) in enumerate(ecarts_zoom.items()):
        ax.plot(serie.index, serie, color=COULEURS_TERMINAL[i], linewidth=2, label=terminal)
    ax.set_ylabel("écart à l'habitude (points)")
    ax.set_title("Part des billets de 10 € sur 24 h glissantes, écart à l'habitude de chaque terminal", pad=20)
    ax.legend(ncol=4, fontsize=8, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d/%m %Hh"))
    for e in ax.get_xticklabels():
        e.set_fontsize(7)
    fig.tight_layout()
    plt.show()